In [1]:
# ============================================================
# 08_AURORA_stronger_financial_baselines_and_purged_allocation.ipynb
# AURORA-TWETF Stronger Financial Baselines and Purged Allocation
#
# Purpose:
# 1. Load purged walk-forward probability outputs from Notebook 07.
# 2. Build AURORA allocation from purged out-of-sample regime probabilities.
# 3. Compare against stronger financial baselines:
#    - equal weight
#    - constrained equal weight
#    - each ETF buy-and-hold
#    - 60/40 ETF-cash
#    - moving-average timing
#    - momentum rotation
#    - inverse-volatility allocation
#    - volatility targeting
#    - minimum-variance allocation
#    - risk-parity proxy
# 4. Evaluate full out-of-sample period and strict test-only period.
# 5. Save paper tables, plots, validation report, and SHA-256 manifest.
#
# Important:
# - Educational/research backtest only.
# - Not personalized financial advice.
# - The AURORA policy here uses purged walk-forward probabilities.
# - Duplicate walk-forward predictions are resolved by keeping the latest fold.
# ============================================================

from __future__ import annotations

import os
import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

try:
    from scipy.optimize import minimize
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
    print("scipy.optimize not available. Minimum-variance baseline will use inverse-variance fallback.")

# ============================================================
# 1. Paths and configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK07_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{NOTEBOOK07_RUN_ID}"

NOTEBOOK08_INPUT_INDEX = (
    NOTEBOOK07_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"
)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "stronger_financial_baselines_purged_allocation" / f"run_{RUN_ID}"

WEIGHT_DIR = RUN_ROOT / "weights"
RETURN_DIR = RUN_ROOT / "returns"
PLOT_DIR = RUN_ROOT / "plots"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAGNOSTIC_DIR = RUN_ROOT / "diagnostics"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    RUN_ROOT,
    WEIGHT_DIR,
    RETURN_DIR,
    PLOT_DIR,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    DIAGNOSTIC_DIR,
    PAPER_FIGURE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Notebook 08: Stronger Baselines and Purged Allocation")
print("=" * 80)
print("Timestamp UTC       :", RUN_TIMESTAMP)
print("Run ID              :", RUN_ID)
print("Notebook 07 root    :", NOTEBOOK07_ROOT)
print("Notebook 08 registry:", NOTEBOOK08_INPUT_INDEX)
print("Run root            :", RUN_ROOT)
print("=" * 80)

if not NOTEBOOK08_INPUT_INDEX.exists():
    raise FileNotFoundError(
        f"Notebook 08 input index not found:\n{NOTEBOOK08_INPUT_INDEX}\n\n"
        "Please run Notebook 07 first."
    )

# ============================================================
# 2. Global allocation settings
# ============================================================

ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"
ALL_ASSETS = ETF_UNIVERSE + [CASH_COL]

CLASS_LABELS = [0, 1, 2, 3, 4]

REGIME_LABELS = {
    0: "Strong Bear",
    1: "Bear",
    2: "Neutral",
    3: "Bull",
    4: "Strong Bull",
}

ANNUALIZATION_DAYS = 252
INITIAL_CAPITAL = 1.0

TRANSACTION_COST_RATE = 0.0010
REBALANCE_FREQUENCY = "monthly"  # stronger comparison: monthly updating

# AURORA baseline configuration.
AURORA_CONFIGS = [
    {
        "policy_name": "AURORA_P0_purged_wf_baseline_60_40",
        "alpha_20d": 0.60,
        "alpha_60d": 0.40,
        "template_variant": "baseline",
        "uncertainty_cash_boost_max": 0.20,
        "max_etf_weight": 0.45,
        "max_00881_weight": 0.35,
        "max_cash_weight": 0.50,
        "min_cash_weight": 0.00,
        "min_confidence_for_full_risk": 0.55,
        "max_confidence_for_min_risk": 0.20,
    },
    {
        "policy_name": "AURORA_P1_purged_wf_more_60d_30_70",
        "alpha_20d": 0.30,
        "alpha_60d": 0.70,
        "template_variant": "baseline",
        "uncertainty_cash_boost_max": 0.20,
        "max_etf_weight": 0.45,
        "max_00881_weight": 0.35,
        "max_cash_weight": 0.50,
        "min_cash_weight": 0.00,
        "min_confidence_for_full_risk": 0.55,
        "max_confidence_for_min_risk": 0.20,
    },
    {
        "policy_name": "AURORA_P2_purged_wf_low_cash_boost",
        "alpha_20d": 0.30,
        "alpha_60d": 0.70,
        "template_variant": "baseline",
        "uncertainty_cash_boost_max": 0.10,
        "max_etf_weight": 0.45,
        "max_00881_weight": 0.35,
        "max_cash_weight": 0.50,
        "min_cash_weight": 0.00,
        "min_confidence_for_full_risk": 0.55,
        "max_confidence_for_min_risk": 0.20,
    },
    {
        "policy_name": "AURORA_P3_purged_wf_no_cash_boost",
        "alpha_20d": 0.30,
        "alpha_60d": 0.70,
        "template_variant": "baseline",
        "uncertainty_cash_boost_max": 0.00,
        "max_etf_weight": 0.45,
        "max_00881_weight": 0.35,
        "max_cash_weight": 0.50,
        "min_cash_weight": 0.00,
        "min_confidence_for_full_risk": 0.55,
        "max_confidence_for_min_risk": 0.20,
    },
]

BASELINE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.15, "006208": 0.25, "00692": 0.25, "00881": 0.00, "CASH": 0.35},
    1: {"0050": 0.25, "006208": 0.30, "00692": 0.30, "00881": 0.05, "CASH": 0.10},
    2: {"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.15, "CASH": 0.00},
    3: {"0050": 0.30, "006208": 0.25, "00692": 0.20, "00881": 0.25, "CASH": 0.00},
    4: {"0050": 0.25, "006208": 0.20, "00692": 0.15, "00881": 0.40, "CASH": 0.00},
}

AGGRESSIVE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.20, "006208": 0.30, "00692": 0.25, "00881": 0.05, "CASH": 0.20},
    1: {"0050": 0.28, "006208": 0.30, "00692": 0.27, "00881": 0.10, "CASH": 0.05},
    2: {"0050": 0.30, "006208": 0.28, "00692": 0.22, "00881": 0.20, "CASH": 0.00},
    3: {"0050": 0.28, "006208": 0.22, "00692": 0.15, "00881": 0.35, "CASH": 0.00},
    4: {"0050": 0.22, "006208": 0.18, "00692": 0.10, "00881": 0.50, "CASH": 0.00},
}

DEFENSIVE_CLASS_WEIGHT_TEMPLATES = {
    0: {"0050": 0.10, "006208": 0.20, "00692": 0.25, "00881": 0.00, "CASH": 0.45},
    1: {"0050": 0.20, "006208": 0.25, "00692": 0.30, "00881": 0.00, "CASH": 0.25},
    2: {"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.05, "CASH": 0.10},
    3: {"0050": 0.32, "006208": 0.28, "00692": 0.25, "00881": 0.15, "CASH": 0.00},
    4: {"0050": 0.30, "006208": 0.25, "00692": 0.20, "00881": 0.25, "CASH": 0.00},
}

TEMPLATE_VARIANTS = {
    "baseline": BASELINE_CLASS_WEIGHT_TEMPLATES,
    "aggressive": AGGRESSIVE_CLASS_WEIGHT_TEMPLATES,
    "defensive": DEFENSIVE_CLASS_WEIGHT_TEMPLATES,
}

LOOKBACK_RETURNS = 63
LOOKBACK_VOL = 63
LOOKBACK_COV = 126
SMA_FAST = 50
SMA_SLOW = 200
VOL_TARGET_ANNUAL = 0.18

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

def clean_symbol_name(x):
    x = str(x)
    x = x.replace(".TW", "")
    x = x.replace(".TWO", "")
    x = x.replace("TW_", "")
    return x

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_table_auto(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass

    df.index.name = "date"
    return df.sort_index()

def load_etf_return_panel():
    candidates = [
        PANEL_DIR / "AURORA_etf_return_panel.parquet",
        PANEL_DIR / "AURORA_etf_returns_panel.parquet",
        PANEL_DIR / "AURORA_return_panel.parquet",
        MODELING_DIR / "AURORA_etf_return_panel.parquet",
    ]

    path = find_first_existing(candidates)

    if path is None:
        raise FileNotFoundError(
            "Could not find ETF return panel. Tried:\n"
            + "\n".join(str(p) for p in candidates)
        )

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df.sort_index()
    df = df.rename(columns={c: clean_symbol_name(c) for c in df.columns})

    missing = [s for s in ETF_UNIVERSE if s not in df.columns]
    if missing:
        raise ValueError(
            f"ETF return panel found at {path}, but missing columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df = df[ETF_UNIVERSE].copy()
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df, path

def load_etf_close_panel(etf_returns):
    candidates = [
        PANEL_DIR / "AURORA_etf_close_panel.parquet",
        PANEL_DIR / "AURORA_close_panel.parquet",
    ]

    path = find_first_existing(candidates)

    if path is not None:
        df = pd.read_parquet(path)
        df.index = pd.to_datetime(df.index)
        df.index.name = "date"
        df = df.sort_index()
        df = df.rename(columns={c: clean_symbol_name(c) for c in df.columns})

        available = [s for s in ETF_UNIVERSE if s in df.columns]
        if len(available) == len(ETF_UNIVERSE):
            return df[ETF_UNIVERSE].copy(), path

    # Fallback: price proxy from returns.
    price_proxy = (1.0 + etf_returns[ETF_UNIVERSE]).cumprod()
    price_proxy = price_proxy / price_proxy.iloc[0] * 100.0
    return price_proxy, None

def normalize_rows(df):
    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[out < 0] = 0.0
    row_sums = out.sum(axis=1)
    zero = row_sums <= 0

    if zero.any():
        out.loc[zero, :] = 1.0 / out.shape[1]
        row_sums = out.sum(axis=1)

    out = out.div(row_sums, axis=0)
    return out

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def proba_cols():
    return [f"proba_class_{i}" for i in CLASS_LABELS]

def probability_features(proba_df):
    cols = proba_cols()
    missing = [c for c in cols if c not in proba_df.columns]
    if missing:
        raise ValueError(f"Missing probability columns: {missing}")

    p = normalize_proba(proba_df[cols].values)

    class_values = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = p @ class_values
    entropy = -np.sum(np.clip(p, 1e-12, 1.0) * np.log(np.clip(p, 1e-12, 1.0)), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))
    sorted_p = np.sort(p, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_variance = (p @ (class_values ** 2)) - expected_class ** 2
    confidence = 1.0 - normalized_entropy

    out = pd.DataFrame(index=proba_df.index)
    out["expected_class"] = expected_class
    out["entropy"] = entropy
    out["normalized_entropy"] = normalized_entropy
    out["confidence_score"] = confidence
    out["probability_margin"] = margin
    out["ordinal_variance"] = ordinal_variance
    out["p_bearish"] = p[:, 0] + p[:, 1]
    out["p_neutral"] = p[:, 2]
    out["p_bullish"] = p[:, 3] + p[:, 4]

    for i, label in enumerate(CLASS_LABELS):
        out[f"proba_class_{label}"] = p[:, i]

    return out

def latest_fold_deduplicate(proba_df, split_filter=None):
    df = proba_df.copy()

    if split_filter is not None and "split" in df.columns:
        if isinstance(split_filter, str):
            split_filter = [split_filter]
        df = df[df["split"].isin(split_filter)].copy()

    if df.empty:
        return df

    df = df.reset_index()

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    df["fold_number"] = (
        df["fold_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(int)
    )

    # Prefer test over validation if both exist for same date, then latest fold.
    split_priority = {"train": 0, "validation": 1, "test": 2}
    if "split" in df.columns:
        df["split_priority"] = df["split"].map(split_priority).fillna(0).astype(int)
    else:
        df["split_priority"] = 0

    df = df.sort_values(["date", "split_priority", "fold_number"])
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.set_index("date").sort_index()
    df.index.name = "date"

    return df.drop(columns=["fold_number", "split_priority"], errors="ignore")

# ============================================================
# 4. AURORA probability-to-weight functions
# ============================================================

def class_template_matrix(template_dict):
    cols = ALL_ASSETS
    mat = np.zeros((len(CLASS_LABELS), len(cols)), dtype=float)

    for c in CLASS_LABELS:
        template = template_dict[c]
        for j, col in enumerate(cols):
            mat[c, j] = float(template.get(col, 0.0))

    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums <= 0] = 1.0
    return mat / row_sums, cols

def apply_weight_constraints(weight_vec, config):
    cols = ALL_ASSETS
    w = pd.Series(weight_vec, index=cols, dtype=float)
    w = w.clip(lower=0.0)

    max_etf_weight = float(config["max_etf_weight"])
    max_00881_weight = float(config["max_00881_weight"])
    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])

    if CASH_COL in w.index:
        w[CASH_COL] = min(max(w[CASH_COL], min_cash_weight), max_cash_weight)

    for etf in ETF_UNIVERSE:
        cap = max_etf_weight
        if etf == "00881":
            cap = min(cap, max_00881_weight)
        w[etf] = min(w[etf], cap)

    total = w.sum()
    if total <= 0:
        w = pd.Series(BASELINE_CLASS_WEIGHT_TEMPLATES[2], dtype=float).reindex(cols).fillna(0.0)
        total = w.sum()

    w = w / total

    for _ in range(10):
        excess = 0.0
        capped = []

        for etf in ETF_UNIVERSE:
            cap = max_etf_weight
            if etf == "00881":
                cap = min(cap, max_00881_weight)

            if w[etf] > cap:
                excess += w[etf] - cap
                w[etf] = cap
                capped.append(etf)

        if w[CASH_COL] > max_cash_weight:
            excess += w[CASH_COL] - max_cash_weight
            w[CASH_COL] = max_cash_weight
            capped.append(CASH_COL)

        if excess <= 1e-12:
            break

        eligible = [c for c in cols if c not in capped]
        if not eligible:
            break

        eligible_sum = w[eligible].sum()
        if eligible_sum <= 0:
            w[eligible] += excess / len(eligible)
        else:
            w[eligible] += excess * (w[eligible] / eligible_sum)

    w = w.clip(lower=0.0)
    total = w.sum()

    if total <= 0:
        w = pd.Series(BASELINE_CLASS_WEIGHT_TEMPLATES[2], dtype=float).reindex(cols).fillna(0.0)
        w = w / w.sum()
    else:
        w = w / total

    return w

def proba_to_template_weights(proba_df, template_dict, config):
    p = normalize_proba(proba_df[proba_cols()].values)
    template_mat, cols = class_template_matrix(template_dict)

    raw = p @ template_mat
    rows = []

    for i in range(raw.shape[0]):
        rows.append(apply_weight_constraints(raw[i, :], config).values)

    return pd.DataFrame(rows, index=proba_df.index, columns=cols)

def blend_weights(w20, w60, config):
    alpha20 = float(config["alpha_20d"])
    alpha60 = float(config["alpha_60d"])

    if alpha20 + alpha60 <= 0:
        alpha20, alpha60 = 0.5, 0.5
    else:
        s = alpha20 + alpha60
        alpha20, alpha60 = alpha20 / s, alpha60 / s

    common = w20.index.intersection(w60.index)
    w = alpha20 * w20.loc[common, ALL_ASSETS] + alpha60 * w60.loc[common, ALL_ASSETS]

    rows = []
    for _, row in w.iterrows():
        rows.append(apply_weight_constraints(row.values, config).values)

    return pd.DataFrame(rows, index=common, columns=ALL_ASSETS)

def apply_uncertainty_gating(weight_df, feature_df, template_dict, config):
    neutral = pd.Series(template_dict[2], dtype=float).reindex(ALL_ASSETS).fillna(0.0)
    neutral = apply_weight_constraints(neutral.values, config)

    min_conf = float(config["min_confidence_for_full_risk"])
    max_conf = float(config["max_confidence_for_min_risk"])
    cash_boost_max = float(config["uncertainty_cash_boost_max"])

    rows = []

    for dt, row in weight_df.iterrows():
        if dt not in feature_df.index:
            rows.append(apply_weight_constraints(row.values, config).values)
            continue

        confidence = float(feature_df.loc[dt, "combined_confidence"])
        p_bearish = float(feature_df.loc[dt, "combined_p_bearish"])
        ord_var = float(feature_df.loc[dt, "combined_ordinal_variance"])

        denom = min_conf - max_conf
        if denom <= 0:
            risk_scale = 1.0
        else:
            risk_scale = (confidence - max_conf) / denom
            risk_scale = float(np.clip(risk_scale, 0.0, 1.0))

        signal_w = pd.Series(row, index=ALL_ASSETS)
        gated = risk_scale * signal_w + (1.0 - risk_scale) * neutral

        uncertainty = 1.0 - confidence
        cash_boost = cash_boost_max * uncertainty * min(1.0, p_bearish + 0.25 * ord_var)
        cash_boost = float(np.clip(cash_boost, 0.0, cash_boost_max))

        if cash_boost > 0:
            pool = gated[ETF_UNIVERSE].sum()
            if pool > 0:
                reduction_ratio = cash_boost / max(pool + cash_boost, 1e-12)
                gated[ETF_UNIVERSE] = gated[ETF_UNIVERSE] * (1.0 - reduction_ratio)
                gated[CASH_COL] = gated[CASH_COL] + cash_boost

        rows.append(apply_weight_constraints(gated.values, config).values)

    return pd.DataFrame(rows, index=weight_df.index, columns=ALL_ASSETS)

def build_aurora_weights(p20, p60, config):
    template_dict = TEMPLATE_VARIANTS[config["template_variant"]]

    common = p20.index.intersection(p60.index)
    p20 = p20.loc[common].copy()
    p60 = p60.loc[common].copy()

    f20 = probability_features(p20)
    f60 = probability_features(p60)

    w20 = proba_to_template_weights(p20, template_dict, config)
    w60 = proba_to_template_weights(p60, template_dict, config)

    raw_w = blend_weights(w20, w60, config)

    alpha20 = float(config["alpha_20d"])
    alpha60 = float(config["alpha_60d"])
    if alpha20 + alpha60 <= 0:
        alpha20, alpha60 = 0.5, 0.5
    else:
        s = alpha20 + alpha60
        alpha20, alpha60 = alpha20 / s, alpha60 / s

    features = pd.DataFrame(index=common)
    features["combined_expected_class"] = alpha20 * f20["expected_class"] + alpha60 * f60["expected_class"]
    features["combined_confidence"] = alpha20 * f20["confidence_score"] + alpha60 * f60["confidence_score"]
    features["combined_p_bearish"] = alpha20 * f20["p_bearish"] + alpha60 * f60["p_bearish"]
    features["combined_p_bullish"] = alpha20 * f20["p_bullish"] + alpha60 * f60["p_bullish"]
    features["combined_ordinal_variance"] = alpha20 * f20["ordinal_variance"] + alpha60 * f60["ordinal_variance"]

    if "split" in p20.columns:
        features["split_20d"] = p20["split"].values
    else:
        features["split_20d"] = "unknown"

    if "split" in p60.columns:
        features["split_60d"] = p60["split"].values
    else:
        features["split_60d"] = "unknown"

    final_w = apply_uncertainty_gating(raw_w, features, template_dict, config)

    return final_w, features

# ============================================================
# 5. Backtest functions
# ============================================================

def get_rebalance_dates(index, frequency):
    idx = pd.DatetimeIndex(index).sort_values()

    if frequency == "monthly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.month])
    elif frequency == "quarterly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.quarter])
    elif frequency == "weekly":
        iso = idx.isocalendar()
        groups = pd.Series(idx, index=idx).groupby([iso.year, iso.week])
    else:
        raise ValueError(f"Unsupported frequency: {frequency}")

    dates = []
    for _, values in groups:
        dates.append(values.iloc[0])

    return pd.DatetimeIndex(dates)

def expand_rebalance_weights_to_daily(signal_weight_df, daily_index, rebalance_dates):
    cols = signal_weight_df.columns.tolist()
    daily = pd.DataFrame(index=daily_index, columns=cols, dtype=float)

    signal_idx = pd.DatetimeIndex(signal_weight_df.index).sort_values()

    for i, reb_date in enumerate(rebalance_dates):
        if i + 1 < len(rebalance_dates):
            period_idx = daily_index[(daily_index >= reb_date) & (daily_index < rebalance_dates[i + 1])]
        else:
            period_idx = daily_index[daily_index >= reb_date]

        prior = signal_idx[signal_idx < reb_date]
        if len(prior) == 0:
            signal_date = signal_idx[0]
        else:
            signal_date = prior[-1]

        daily.loc[period_idx, :] = signal_weight_df.loc[signal_date].values

    daily = daily.ffill().bfill()
    return daily

def compute_turnover(daily_weights, rebalance_dates):
    turnover = pd.Series(0.0, index=daily_weights.index)
    prev_w = None

    for dt in rebalance_dates:
        if dt not in daily_weights.index:
            continue

        w = daily_weights.loc[dt, ALL_ASSETS]

        if prev_w is None:
            turnover.loc[dt] = w.drop(labels=[CASH_COL], errors="ignore").abs().sum()
        else:
            turnover.loc[dt] = (w - prev_w).abs().sum() / 2.0

        prev_w = w

    return turnover

def backtest_policy(policy_name, signal_weights, etf_returns):
    returns = etf_returns.copy()
    returns[CASH_COL] = 0.0

    signal_weights = signal_weights.reindex(columns=ALL_ASSETS).fillna(0.0)
    signal_weights = normalize_rows(signal_weights)

    common = returns.index.intersection(signal_weights.index)
    returns = returns.loc[common, ALL_ASSETS].copy()
    signal_weights = signal_weights.loc[common, ALL_ASSETS].copy()

    if len(common) == 0:
        raise ValueError(f"No common dates for policy {policy_name}")

    rebalance_dates = get_rebalance_dates(returns.index, REBALANCE_FREQUENCY)

    daily_weights = expand_rebalance_weights_to_daily(
        signal_weight_df=signal_weights,
        daily_index=returns.index,
        rebalance_dates=rebalance_dates,
    )

    daily_weights = daily_weights.reindex(columns=ALL_ASSETS).fillna(0.0)
    daily_weights = normalize_rows(daily_weights)

    gross_return = (daily_weights[ALL_ASSETS] * returns[ALL_ASSETS]).sum(axis=1)
    turnover = compute_turnover(daily_weights, rebalance_dates)
    transaction_cost = turnover * TRANSACTION_COST_RATE
    net_return = gross_return - transaction_cost

    equity = (1.0 + net_return).cumprod() * INITIAL_CAPITAL

    out = pd.DataFrame(index=returns.index)
    out.index.name = "date"
    out["policy_name"] = policy_name
    out["gross_return"] = gross_return
    out["turnover"] = turnover
    out["transaction_cost"] = transaction_cost
    out["net_return"] = net_return
    out["equity"] = equity
    out["drawdown"] = equity / equity.cummax() - 1.0
    out["is_rebalance_date"] = out.index.isin(rebalance_dates)

    return out, daily_weights

def rebase_return_df(return_df):
    out = return_df.copy()
    out["equity"] = (1.0 + out["net_return"]).cumprod()
    out["drawdown"] = out["equity"] / out["equity"].cummax() - 1.0
    return out

def performance_metrics(return_df):
    r = return_df["net_return"].astype(float)
    equity = return_df["equity"].astype(float)
    drawdown = return_df["drawdown"].astype(float)

    n = len(r)
    if n == 0:
        return {}

    total_return = float(equity.iloc[-1] / equity.iloc[0] - 1.0) if equity.iloc[0] != 0 else np.nan
    annual_return = float((1.0 + total_return) ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)
    annual_vol = float(r.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if n > 1 else np.nan
    sharpe = annual_return / annual_vol if annual_vol and annual_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if len(downside) > 1 else np.nan
    sortino = annual_return / downside_vol if downside_vol and downside_vol > 0 else np.nan

    max_drawdown = float(drawdown.min())
    calmar = annual_return / abs(max_drawdown) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": float((r > 0).mean()),
        "avg_daily_return": float(r.mean()),
        "avg_turnover": float(return_df["turnover"].mean()),
        "total_turnover": float(return_df["turnover"].sum()),
        "total_transaction_cost": float(return_df["transaction_cost"].sum()),
        "final_equity": float(equity.iloc[-1]),
    }

# ============================================================
# 6. Stronger financial baseline weight builders
# ============================================================

def constant_weights(index, weights_dict):
    df = pd.DataFrame(0.0, index=index, columns=ALL_ASSETS)
    for k, v in weights_dict.items():
        if k in df.columns:
            df[k] = float(v)
    return normalize_rows(df)

def rolling_momentum_rotation(etf_returns, top_k=1, lookback=63, cash_weight=0.0):
    mom = (1.0 + etf_returns[ETF_UNIVERSE]).rolling(lookback).apply(np.prod, raw=True) - 1.0

    weights = pd.DataFrame(0.0, index=etf_returns.index, columns=ALL_ASSETS)

    for dt in etf_returns.index:
        if dt not in mom.index or mom.loc[dt].isna().all():
            weights.loc[dt, ETF_UNIVERSE] = 1.0 / len(ETF_UNIVERSE)
            weights.loc[dt, CASH_COL] = 0.0
            continue

        ranked = mom.loc[dt].sort_values(ascending=False)
        selected = ranked.head(top_k).index.tolist()

        weights.loc[dt, CASH_COL] = cash_weight
        for s in selected:
            weights.loc[dt, s] = (1.0 - cash_weight) / top_k

    return normalize_rows(weights)

def moving_average_timing(etf_close, base_weights, fast=50, slow=200, defensive_cash=0.50):
    close = etf_close[ETF_UNIVERSE].copy()
    market_proxy = close["0050"].copy()

    sma_fast = market_proxy.rolling(fast).mean()
    sma_slow = market_proxy.rolling(slow).mean()
    risk_on = sma_fast > sma_slow

    base = constant_weights(close.index, base_weights)
    weights = base.copy()

    for dt in weights.index:
        if pd.isna(risk_on.loc[dt]):
            continue

        if not bool(risk_on.loc[dt]):
            etf_scale = 1.0 - defensive_cash
            weights.loc[dt, ETF_UNIVERSE] = base.loc[dt, ETF_UNIVERSE] * etf_scale
            weights.loc[dt, CASH_COL] = defensive_cash

    return normalize_rows(weights)

def inverse_volatility_weights(etf_returns, lookback=63, cash_weight=0.0):
    vol = etf_returns[ETF_UNIVERSE].rolling(lookback).std()

    weights = pd.DataFrame(0.0, index=etf_returns.index, columns=ALL_ASSETS)

    for dt in etf_returns.index:
        v = vol.loc[dt]

        if v.isna().any() or (v <= 0).all():
            weights.loc[dt, ETF_UNIVERSE] = (1.0 - cash_weight) / len(ETF_UNIVERSE)
            weights.loc[dt, CASH_COL] = cash_weight
            continue

        inv = 1.0 / v.replace(0.0, np.nan)
        inv = inv.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        if inv.sum() <= 0:
            weights.loc[dt, ETF_UNIVERSE] = (1.0 - cash_weight) / len(ETF_UNIVERSE)
        else:
            weights.loc[dt, ETF_UNIVERSE] = (1.0 - cash_weight) * inv / inv.sum()

        weights.loc[dt, CASH_COL] = cash_weight

    return normalize_rows(weights)

def volatility_target_weights(etf_returns, base_weights, lookback=63, target_vol=0.18):
    base = base_weights.reindex(index=etf_returns.index).ffill().bfill()
    base = base.reindex(columns=ALL_ASSETS).fillna(0.0)
    base = normalize_rows(base)

    asset_returns = etf_returns.copy()
    asset_returns[CASH_COL] = 0.0

    base_port_r = (base[ALL_ASSETS] * asset_returns[ALL_ASSETS]).sum(axis=1)
    realized_vol = base_port_r.rolling(lookback).std() * np.sqrt(ANNUALIZATION_DAYS)

    weights = base.copy()

    for dt in weights.index:
        rv = realized_vol.loc[dt]

        if pd.isna(rv) or rv <= 0:
            scale = 1.0
        else:
            scale = min(1.0, target_vol / rv)

        weights.loc[dt, ETF_UNIVERSE] = base.loc[dt, ETF_UNIVERSE] * scale
        weights.loc[dt, CASH_COL] = 1.0 - weights.loc[dt, ETF_UNIVERSE].sum()

    return normalize_rows(weights)

def min_variance_weights(etf_returns, lookback=126):
    weights = pd.DataFrame(0.0, index=etf_returns.index, columns=ALL_ASSETS)

    for i, dt in enumerate(etf_returns.index):
        if i < lookback:
            weights.loc[dt, ETF_UNIVERSE] = 1.0 / len(ETF_UNIVERSE)
            continue

        window = etf_returns[ETF_UNIVERSE].iloc[i - lookback:i].dropna()

        if len(window) < 30:
            weights.loc[dt, ETF_UNIVERSE] = 1.0 / len(ETF_UNIVERSE)
            continue

        cov = window.cov().values
        cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)
        cov = cov + np.eye(len(ETF_UNIVERSE)) * 1e-8

        if HAS_SCIPY:
            n = len(ETF_UNIVERSE)

            def obj(w):
                return float(w.T @ cov @ w)

            cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
            bounds = [(0.0, 1.0) for _ in range(n)]
            x0 = np.ones(n) / n

            res = minimize(obj, x0=x0, bounds=bounds, constraints=cons, method="SLSQP")

            if res.success:
                w = np.clip(res.x, 0.0, 1.0)
                w = w / w.sum()
            else:
                var = np.diag(cov)
                inv = 1.0 / np.maximum(var, 1e-12)
                w = inv / inv.sum()
        else:
            var = np.diag(cov)
            inv = 1.0 / np.maximum(var, 1e-12)
            w = inv / inv.sum()

        weights.loc[dt, ETF_UNIVERSE] = w

    weights[CASH_COL] = 0.0
    return normalize_rows(weights)

def risk_parity_proxy_weights(etf_returns, lookback=126):
    # Inverse-volatility proxy for equal-risk contribution.
    return inverse_volatility_weights(etf_returns, lookback=lookback, cash_weight=0.0)

# ============================================================
# 7. Plotting functions
# ============================================================

def plot_equity_curves(all_returns_df, path, title, top_n=None):
    final_equity = (
        all_returns_df.groupby("policy_name")["equity"]
        .last()
        .sort_values(ascending=False)
    )

    policies = final_equity.index.tolist()
    if top_n is not None:
        policies = policies[:top_n]

    plt.figure(figsize=(12, 6))
    for policy in policies:
        grp = all_returns_df[all_returns_df["policy_name"] == policy].sort_index()
        plt.plot(grp.index, grp["equity"], label=policy, linewidth=1.7)

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Equity, initial capital = 1")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_drawdowns(all_returns_df, path, title, policies=None):
    if policies is None:
        policies = all_returns_df["policy_name"].unique().tolist()

    plt.figure(figsize=(12, 6))
    for policy in policies:
        grp = all_returns_df[all_returns_df["policy_name"] == policy].sort_index()
        plt.plot(grp.index, grp["drawdown"], label=policy, linewidth=1.5)

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_metric_bar(df, metric, path, title, higher_is_better=True, top_n=None):
    tmp = df.copy()
    tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=[metric])
    tmp = tmp.sort_values(metric, ascending=not higher_is_better)

    if top_n is not None:
        tmp = tmp.head(top_n)

    plt.figure(figsize=(11, max(4, 0.4 * len(tmp))))
    sns.barplot(data=tmp, y="policy_name", x=metric, color="#4C72B0")
    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_weights_area(weight_df, policy_name, path):
    df = weight_df[ALL_ASSETS].copy()

    plt.figure(figsize=(12, 6))
    plt.stackplot(df.index, [df[c].values for c in ALL_ASSETS], labels=ALL_ASSETS, alpha=0.85)
    plt.title(f"Weights: {policy_name}")
    plt.xlabel("Date")
    plt.ylabel("Weight")
    plt.ylim(0, 1)
    plt.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

# ============================================================
# 8. Load data and purged walk-forward probabilities
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading ETF panels and Notebook 07 probabilities")
print("=" * 80)

input_index_df = pd.read_csv(NOTEBOOK08_INPUT_INDEX)

etf_returns, etf_return_path = load_etf_return_panel()
etf_close, etf_close_path = load_etf_close_panel(etf_returns)

print("ETF return panel:", etf_return_path)
print("ETF return shape:", etf_returns.shape)
print("ETF date range  :", etf_returns.index.min().date(), "to", etf_returns.index.max().date())

if etf_close_path is not None:
    print("ETF close panel :", etf_close_path)
else:
    print("ETF close panel : price proxy from ETF returns")

print("\nNotebook 08 input index:")
print(input_index_df.to_string(index=False))

p20_path = None
p60_path = None

for _, row in input_index_df.iterrows():
    target_col = row["target_col"]
    path = Path(row["probability_path_parquet"])
    if not path.exists():
        path = Path(row["probability_path_csv"])

    if "20d" in target_col:
        p20_path = path
    elif "60d" in target_col:
        p60_path = path

if p20_path is None or p60_path is None:
    raise ValueError("Could not locate both 20d and 60d probability files from Notebook 07 index.")

p20_raw = read_table_auto(p20_path)
p60_raw = read_table_auto(p60_path)

p20_oos = latest_fold_deduplicate(p20_raw, split_filter=["validation", "test"])
p60_oos = latest_fold_deduplicate(p60_raw, split_filter=["validation", "test"])

p20_test = latest_fold_deduplicate(p20_raw, split_filter=["test"])
p60_test = latest_fold_deduplicate(p60_raw, split_filter=["test"])

print("\nLoaded probabilities:")
print("20d raw :", p20_raw.shape, p20_path)
print("60d raw :", p60_raw.shape, p60_path)
print("20d OOS :", p20_oos.shape, p20_oos.index.min().date(), "to", p20_oos.index.max().date())
print("60d OOS :", p60_oos.shape, p60_oos.index.min().date(), "to", p60_oos.index.max().date())
print("20d test:", p20_test.shape, p20_test.index.min().date(), "to", p20_test.index.max().date())
print("60d test:", p60_test.shape, p60_test.index.min().date(), "to", p60_test.index.max().date())

# Define evaluation date sets.
oos_dates = p20_oos.index.intersection(p60_oos.index).intersection(etf_returns.index)
test_dates = p20_test.index.intersection(p60_test.index).intersection(etf_returns.index)

print("\nEvaluation periods:")
print("OOS dates :", len(oos_dates), oos_dates.min().date(), "to", oos_dates.max().date())
print("Test dates:", len(test_dates), test_dates.min().date(), "to", test_dates.max().date())

# Restrict data to OOS period for all policy construction.
etf_returns_oos = etf_returns.loc[oos_dates.min():oos_dates.max()].copy()
etf_close_oos = etf_close.loc[etf_returns_oos.index].copy()

# ============================================================
# 9. Build AURORA purged allocation policies
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Building AURORA purged walk-forward allocation policies")
print("=" * 80)

policy_weight_dict = {}
policy_type_dict = {}
policy_description_rows = {}
aurora_feature_frames = []

for config in AURORA_CONFIGS:
    policy_name = config["policy_name"]

    weights, features = build_aurora_weights(
        p20=p20_oos,
        p60=p60_oos,
        config=config,
    )

    weights = weights.loc[weights.index.intersection(etf_returns_oos.index)].copy()
    features = features.loc[weights.index].copy()

    policy_weight_dict[policy_name] = weights
    policy_type_dict[policy_name] = "AURORA_purged_walk_forward"

    desc = {
        "policy_name": policy_name,
        "policy_type": "AURORA_purged_walk_forward",
        "alpha_20d": config["alpha_20d"],
        "alpha_60d": config["alpha_60d"],
        "template_variant": config["template_variant"],
        "uncertainty_cash_boost_max": config["uncertainty_cash_boost_max"],
        "max_etf_weight": config["max_etf_weight"],
        "max_00881_weight": config["max_00881_weight"],
        "max_cash_weight": config["max_cash_weight"],
        "description": "AURORA regime probability allocation using purged walk-forward selected ensemble probabilities.",
    }

    policy_description_rows[policy_name] = desc

    features_out = features.copy()
    features_out.insert(0, "policy_name", policy_name)
    aurora_feature_frames.append(features_out)

    weights.to_parquet(WEIGHT_DIR / f"signal_weights_{safe_name(policy_name)}.parquet")
    weights.to_csv(WEIGHT_DIR / f"signal_weights_{safe_name(policy_name)}.csv")

    features_out.to_parquet(DIAGNOSTIC_DIR / f"signal_features_{safe_name(policy_name)}.parquet")
    features_out.to_csv(DIAGNOSTIC_DIR / f"signal_features_{safe_name(policy_name)}.csv")

    print(policy_name, weights.shape, weights.index.min().date(), "to", weights.index.max().date())

aurora_features_all_df = pd.concat(aurora_feature_frames, axis=0)
aurora_features_all_df.to_csv(DIAGNOSTIC_DIR / "aurora_signal_features_all.csv")

# ============================================================
# 10. Build stronger financial baselines
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Building stronger financial baselines")
print("=" * 80)

idx = etf_returns_oos.index

baseline_specs = {
    "B1_equal_weight_all_etfs": {
        "weights": {"0050": 0.25, "006208": 0.25, "00692": 0.25, "00881": 0.25, "CASH": 0.0},
        "description": "Passive equal-weight allocation across all four ETFs.",
    },
    "B2_equal_weight_constrained": {
        "weights": {"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.15, "CASH": 0.0},
        "description": "Passive constrained allocation with lower semiconductor ETF weight.",
    },
    "B3_0050_only": {
        "weights": {"0050": 1.0, "006208": 0.0, "00692": 0.0, "00881": 0.0, "CASH": 0.0},
        "description": "Buy-and-hold 0050 benchmark.",
    },
    "B4_006208_only": {
        "weights": {"0050": 0.0, "006208": 1.0, "00692": 0.0, "00881": 0.0, "CASH": 0.0},
        "description": "Buy-and-hold 006208 benchmark.",
    },
    "B5_00692_only": {
        "weights": {"0050": 0.0, "006208": 0.0, "00692": 1.0, "00881": 0.0, "CASH": 0.0},
        "description": "Buy-and-hold 00692 benchmark.",
    },
    "B6_00881_only": {
        "weights": {"0050": 0.0, "006208": 0.0, "00692": 0.0, "00881": 1.0, "CASH": 0.0},
        "description": "Buy-and-hold 00881 semiconductor ETF benchmark.",
    },
    "B7_equal_weight_60_40_cash": {
        "weights": {"0050": 0.15, "006208": 0.15, "00692": 0.15, "00881": 0.15, "CASH": 0.40},
        "description": "60% equal-weight ETF and 40% cash defensive benchmark.",
    },
    "B8_constrained_80_20_cash": {
        "weights": {"0050": 0.24, "006208": 0.24, "00692": 0.20, "00881": 0.12, "CASH": 0.20},
        "description": "80% constrained ETF and 20% cash benchmark.",
    },
}

for name, spec in baseline_specs.items():
    w = constant_weights(idx, spec["weights"])
    policy_weight_dict[name] = w
    policy_type_dict[name] = "passive_benchmark"

    policy_description_rows[name] = {
        "policy_name": name,
        "policy_type": "passive_benchmark",
        "description": spec["description"],
    }

# Dynamic baselines.
dynamic_baselines = {}

dynamic_baselines["B9_momentum_top1_63d"] = rolling_momentum_rotation(
    etf_returns_oos,
    top_k=1,
    lookback=LOOKBACK_RETURNS,
    cash_weight=0.0,
)

dynamic_baselines["B10_momentum_top2_63d"] = rolling_momentum_rotation(
    etf_returns_oos,
    top_k=2,
    lookback=LOOKBACK_RETURNS,
    cash_weight=0.0,
)

dynamic_baselines["B11_momentum_top1_63d_20pct_cash"] = rolling_momentum_rotation(
    etf_returns_oos,
    top_k=1,
    lookback=LOOKBACK_RETURNS,
    cash_weight=0.20,
)

dynamic_baselines["B12_ma_timing_equal_weight"] = moving_average_timing(
    etf_close_oos,
    base_weights={"0050": 0.25, "006208": 0.25, "00692": 0.25, "00881": 0.25, "CASH": 0.0},
    fast=SMA_FAST,
    slow=SMA_SLOW,
    defensive_cash=0.50,
)

dynamic_baselines["B13_ma_timing_constrained"] = moving_average_timing(
    etf_close_oos,
    base_weights={"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.15, "CASH": 0.0},
    fast=SMA_FAST,
    slow=SMA_SLOW,
    defensive_cash=0.50,
)

dynamic_baselines["B14_inverse_volatility_63d"] = inverse_volatility_weights(
    etf_returns_oos,
    lookback=LOOKBACK_VOL,
    cash_weight=0.0,
)

dynamic_baselines["B15_inverse_volatility_63d_20pct_cash"] = inverse_volatility_weights(
    etf_returns_oos,
    lookback=LOOKBACK_VOL,
    cash_weight=0.20,
)

minvar_w = min_variance_weights(etf_returns_oos, lookback=LOOKBACK_COV)
dynamic_baselines["B16_minimum_variance_126d"] = minvar_w

risk_parity_w = risk_parity_proxy_weights(etf_returns_oos, lookback=LOOKBACK_COV)
dynamic_baselines["B17_risk_parity_proxy_126d"] = risk_parity_w

base_equal = constant_weights(
    idx,
    {"0050": 0.25, "006208": 0.25, "00692": 0.25, "00881": 0.25, "CASH": 0.0},
)
dynamic_baselines["B18_vol_target_equal_weight_18pct"] = volatility_target_weights(
    etf_returns_oos,
    base_weights=base_equal,
    lookback=LOOKBACK_VOL,
    target_vol=VOL_TARGET_ANNUAL,
)

base_constrained = constant_weights(
    idx,
    {"0050": 0.30, "006208": 0.30, "00692": 0.25, "00881": 0.15, "CASH": 0.0},
)
dynamic_baselines["B19_vol_target_constrained_18pct"] = volatility_target_weights(
    etf_returns_oos,
    base_weights=base_constrained,
    lookback=LOOKBACK_VOL,
    target_vol=VOL_TARGET_ANNUAL,
)

for name, w in dynamic_baselines.items():
    policy_weight_dict[name] = w
    policy_type_dict[name] = "dynamic_financial_baseline"

    policy_description_rows[name] = {
        "policy_name": name,
        "policy_type": "dynamic_financial_baseline",
        "description": "Dynamic financial baseline generated from trailing returns, volatility, trend, or covariance estimates.",
    }

for name, w in policy_weight_dict.items():
    w.to_parquet(WEIGHT_DIR / f"signal_weights_{safe_name(name)}.parquet")
    w.to_csv(WEIGHT_DIR / f"signal_weights_{safe_name(name)}.csv")

print("Total policies:", len(policy_weight_dict))
print(pd.Series(policy_type_dict).value_counts().to_string())

# ============================================================
# 11. Backtest all policies
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Backtesting AURORA and stronger baselines")
print("=" * 80)

all_return_frames = []
all_weight_frames = []
metric_rows = []
test_metric_rows = []

for policy_name, signal_w in policy_weight_dict.items():
    print("Backtesting:", policy_name)

    returns_df, daily_weights = backtest_policy(
        policy_name=policy_name,
        signal_weights=signal_w,
        etf_returns=etf_returns_oos,
    )

    m = performance_metrics(returns_df)
    m["run_id"] = RUN_ID
    m["policy_name"] = policy_name
    m["policy_type"] = policy_type_dict[policy_name]
    m["period"] = "oos_validation_and_test"
    m["rebalance_frequency"] = REBALANCE_FREQUENCY
    m["transaction_cost_rate"] = TRANSACTION_COST_RATE
    metric_rows.append(m)

    # Strict test-only period from purged walk-forward common test dates.
    test_idx = returns_df.index.intersection(test_dates)
    if len(test_idx) > 0:
        test_df = returns_df.loc[test_idx].copy()
        test_df = rebase_return_df(test_df)

        tm = performance_metrics(test_df)
        tm["run_id"] = RUN_ID
        tm["policy_name"] = policy_name
        tm["policy_type"] = policy_type_dict[policy_name]
        tm["period"] = "strict_test_only"
        tm["rebalance_frequency"] = REBALANCE_FREQUENCY
        tm["transaction_cost_rate"] = TRANSACTION_COST_RATE
        test_metric_rows.append(tm)

    rf = returns_df.copy()
    rf.index.name = "date"
    all_return_frames.append(rf)

    wf = daily_weights.copy()
    wf.index.name = "date"
    wf.insert(0, "policy_name", policy_name)
    all_weight_frames.append(wf)

    returns_df.to_parquet(RETURN_DIR / f"returns_{safe_name(policy_name)}.parquet")
    returns_df.to_csv(RETURN_DIR / f"returns_{safe_name(policy_name)}.csv")

    daily_weights.to_parquet(WEIGHT_DIR / f"daily_weights_{safe_name(policy_name)}.parquet")
    daily_weights.to_csv(WEIGHT_DIR / f"daily_weights_{safe_name(policy_name)}.csv")

all_returns_df = pd.concat(all_return_frames, axis=0)
all_weights_df = pd.concat(all_weight_frames, axis=0)

performance_df = pd.DataFrame(metric_rows)
test_performance_df = pd.DataFrame(test_metric_rows)

performance_df = performance_df.sort_values(["sharpe_ratio", "final_equity"], ascending=[False, False])
test_performance_df = test_performance_df.sort_values(["sharpe_ratio", "final_equity"], ascending=[False, False])

all_returns_df.to_parquet(RETURN_DIR / "returns_all_policies.parquet")
all_returns_df.to_csv(RETURN_DIR / "returns_all_policies.csv")

all_weights_df.to_parquet(WEIGHT_DIR / "daily_weights_all_policies.parquet")
all_weights_df.to_csv(WEIGHT_DIR / "daily_weights_all_policies.csv")

performance_df.to_csv(TABLE_RUN_DIR / "allocation_performance_oos_validation_and_test.csv", index=False)
performance_df.to_csv(TABLE_DIR / f"table_59_stronger_baseline_allocation_performance_oos_{RUN_ID}.csv", index=False)

test_performance_df.to_csv(TABLE_RUN_DIR / "allocation_performance_strict_test_only.csv", index=False)
test_performance_df.to_csv(TABLE_DIR / f"table_60_stronger_baseline_allocation_performance_test_only_{RUN_ID}.csv", index=False)

print("\nOOS validation+test performance:")
print(performance_df[[
    "policy_name",
    "policy_type",
    "n_days",
    "start_date",
    "end_date",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "final_equity",
]].head(30).to_string(index=False))

print("\nStrict test-only performance:")
print(test_performance_df[[
    "policy_name",
    "policy_type",
    "n_days",
    "start_date",
    "end_date",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "final_equity",
]].head(30).to_string(index=False))

# ============================================================
# 12. Ranking and comparison tables
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Creating rankings and comparison tables")
print("=" * 80)

def add_composite_rank(df):
    out = df.copy()

    out["rank_total_return"] = out["total_return"].rank(ascending=False, method="min")
    out["rank_sharpe"] = out["sharpe_ratio"].rank(ascending=False, method="min")
    out["rank_sortino"] = out["sortino_ratio"].rank(ascending=False, method="min")
    out["rank_drawdown"] = out["max_drawdown"].rank(ascending=False, method="min")
    out["rank_calmar"] = out["calmar_ratio"].rank(ascending=False, method="min")

    out["allocation_composite_rank"] = (
        out["rank_total_return"]
        + out["rank_sharpe"]
        + out["rank_sortino"]
        + out["rank_drawdown"]
        + out["rank_calmar"]
    ) / 5.0

    out = out.sort_values(
        ["allocation_composite_rank", "sharpe_ratio", "total_return"],
        ascending=[True, False, False],
    )

    return out

rank_oos_df = add_composite_rank(performance_df)
rank_test_df = add_composite_rank(test_performance_df)

rank_oos_df.to_csv(TABLE_RUN_DIR / "allocation_rankings_oos_validation_and_test.csv", index=False)
rank_oos_df.to_csv(TABLE_DIR / f"table_61_allocation_rankings_oos_{RUN_ID}.csv", index=False)

rank_test_df.to_csv(TABLE_RUN_DIR / "allocation_rankings_strict_test_only.csv", index=False)
rank_test_df.to_csv(TABLE_DIR / f"table_62_allocation_rankings_test_only_{RUN_ID}.csv", index=False)

# Active AURORA vs best benchmark.
best_benchmark_oos = (
    rank_oos_df[rank_oos_df["policy_type"].isin(["passive_benchmark", "dynamic_financial_baseline"])]
    .sort_values("allocation_composite_rank")
    .iloc[0]
)

aurora_vs_benchmark_rows = []

for _, row in rank_oos_df[rank_oos_df["policy_type"] == "AURORA_purged_walk_forward"].iterrows():
    aurora_vs_benchmark_rows.append({
        "period": "oos_validation_and_test",
        "aurora_policy": row["policy_name"],
        "benchmark_policy": best_benchmark_oos["policy_name"],
        "aurora_total_return": row["total_return"],
        "benchmark_total_return": best_benchmark_oos["total_return"],
        "excess_total_return": row["total_return"] - best_benchmark_oos["total_return"],
        "aurora_sharpe": row["sharpe_ratio"],
        "benchmark_sharpe": best_benchmark_oos["sharpe_ratio"],
        "excess_sharpe": row["sharpe_ratio"] - best_benchmark_oos["sharpe_ratio"],
        "aurora_max_drawdown": row["max_drawdown"],
        "benchmark_max_drawdown": best_benchmark_oos["max_drawdown"],
        "drawdown_improvement": row["max_drawdown"] - best_benchmark_oos["max_drawdown"],
    })

if not rank_test_df.empty:
    best_benchmark_test = (
        rank_test_df[rank_test_df["policy_type"].isin(["passive_benchmark", "dynamic_financial_baseline"])]
        .sort_values("allocation_composite_rank")
        .iloc[0]
    )

    for _, row in rank_test_df[rank_test_df["policy_type"] == "AURORA_purged_walk_forward"].iterrows():
        aurora_vs_benchmark_rows.append({
            "period": "strict_test_only",
            "aurora_policy": row["policy_name"],
            "benchmark_policy": best_benchmark_test["policy_name"],
            "aurora_total_return": row["total_return"],
            "benchmark_total_return": best_benchmark_test["total_return"],
            "excess_total_return": row["total_return"] - best_benchmark_test["total_return"],
            "aurora_sharpe": row["sharpe_ratio"],
            "benchmark_sharpe": best_benchmark_test["sharpe_ratio"],
            "excess_sharpe": row["sharpe_ratio"] - best_benchmark_test["sharpe_ratio"],
            "aurora_max_drawdown": row["max_drawdown"],
            "benchmark_max_drawdown": best_benchmark_test["max_drawdown"],
            "drawdown_improvement": row["max_drawdown"] - best_benchmark_test["max_drawdown"],
        })

aurora_vs_benchmark_df = pd.DataFrame(aurora_vs_benchmark_rows)
aurora_vs_benchmark_df.to_csv(TABLE_RUN_DIR / "aurora_vs_best_benchmark.csv", index=False)
aurora_vs_benchmark_df.to_csv(TABLE_DIR / f"table_63_aurora_vs_best_benchmark_{RUN_ID}.csv", index=False)

policy_description_df = pd.DataFrame(list(policy_description_rows.values()))
policy_description_df.to_csv(TABLE_RUN_DIR / "policy_descriptions.csv", index=False)
policy_description_df.to_csv(TABLE_DIR / f"table_64_policy_descriptions_stronger_baselines_{RUN_ID}.csv", index=False)

print("\nTop OOS rankings:")
print(rank_oos_df[[
    "policy_name",
    "policy_type",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "allocation_composite_rank",
]].head(25).to_string(index=False))

print("\nAURORA vs best benchmark:")
print(aurora_vs_benchmark_df.to_string(index=False))

# ============================================================
# 13. Plots
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Creating plots and paper figures")
print("=" * 80)

plot_equity_curves(
    all_returns_df,
    path=PLOT_DIR / "equity_curves_top12_oos.png",
    title="Top OOS equity curves: AURORA vs stronger financial baselines",
    top_n=12,
)

top_policies = rank_oos_df.head(12)["policy_name"].tolist()

plot_drawdowns(
    all_returns_df,
    path=PLOT_DIR / "drawdowns_top12_oos.png",
    title="Top OOS drawdowns: AURORA vs stronger financial baselines",
    policies=top_policies,
)

plot_metric_bar(
    rank_oos_df,
    metric="sharpe_ratio",
    path=PLOT_DIR / "oos_sharpe_by_policy.png",
    title="OOS Sharpe ratio by policy",
    higher_is_better=True,
    top_n=25,
)

plot_metric_bar(
    rank_oos_df,
    metric="total_return",
    path=PLOT_DIR / "oos_total_return_by_policy.png",
    title="OOS total return by policy",
    higher_is_better=True,
    top_n=25,
)

plot_metric_bar(
    rank_oos_df,
    metric="max_drawdown",
    path=PLOT_DIR / "oos_max_drawdown_by_policy.png",
    title="OOS max drawdown by policy",
    higher_is_better=True,
    top_n=25,
)

if not rank_test_df.empty:
    plot_metric_bar(
        rank_test_df,
        metric="sharpe_ratio",
        path=PLOT_DIR / "test_only_sharpe_by_policy.png",
        title="Strict test-only Sharpe ratio by policy",
        higher_is_better=True,
        top_n=25,
    )

    plot_metric_bar(
        rank_test_df,
        metric="total_return",
        path=PLOT_DIR / "test_only_total_return_by_policy.png",
        title="Strict test-only total return by policy",
        higher_is_better=True,
        top_n=25,
    )

# Weight plots for AURORA and best benchmark.
for policy_name in list(policy_weight_dict.keys()):
    if policy_name.startswith("AURORA") or policy_name in top_policies[:5]:
        wf = all_weights_df[all_weights_df["policy_name"] == policy_name].copy()
        if not wf.empty:
            wf = wf.drop(columns=["policy_name"], errors="ignore")
            plot_weights_area(
                wf,
                policy_name=policy_name,
                path=PLOT_DIR / f"weights_area_{safe_name(policy_name)}.png",
            )

# Copy paper-selected figures.
paper_figures = [
    "equity_curves_top12_oos.png",
    "drawdowns_top12_oos.png",
    "oos_sharpe_by_policy.png",
    "oos_total_return_by_policy.png",
    "oos_max_drawdown_by_policy.png",
    "test_only_sharpe_by_policy.png",
    "test_only_total_return_by_policy.png",
]

for fname in paper_figures:
    src = PLOT_DIR / fname
    if src.exists():
        dst = PAPER_FIGURE_DIR / fname
        dst.write_bytes(src.read_bytes())

        global_dst = FIGURE_DIR / f"{Path(fname).stem}_{RUN_ID}.png"
        global_dst.write_bytes(src.read_bytes())

print("Plots saved to:", PLOT_DIR)
print("Paper figures saved to:", PAPER_FIGURE_DIR)

# ============================================================
# 14. Diagnostic summary
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Creating diagnostic summary")
print("=" * 80)

diagnostic_rows = []

best_aurora_oos = (
    rank_oos_df[rank_oos_df["policy_type"] == "AURORA_purged_walk_forward"]
    .sort_values("allocation_composite_rank")
    .iloc[0]
)

best_all_oos = rank_oos_df.sort_values("allocation_composite_rank").iloc[0]

diagnostic_rows.append({
    "question": "Does best AURORA policy beat best stronger financial baseline in OOS composite rank?",
    "finding": "Yes" if best_aurora_oos["allocation_composite_rank"] < best_benchmark_oos["allocation_composite_rank"] else "No",
    "evidence": (
        f"Best AURORA={best_aurora_oos['policy_name']}, rank={best_aurora_oos['allocation_composite_rank']:.2f}, "
        f"Sharpe={best_aurora_oos['sharpe_ratio']:.4f}; "
        f"Best benchmark={best_benchmark_oos['policy_name']}, rank={best_benchmark_oos['allocation_composite_rank']:.2f}, "
        f"Sharpe={best_benchmark_oos['sharpe_ratio']:.4f}."
    ),
})

diagnostic_rows.append({
    "question": "Which policy is best overall in OOS composite rank?",
    "finding": str(best_all_oos["policy_name"]),
    "evidence": (
        f"Policy type={best_all_oos['policy_type']}, "
        f"total_return={best_all_oos['total_return']:.4f}, "
        f"Sharpe={best_all_oos['sharpe_ratio']:.4f}, "
        f"max_drawdown={best_all_oos['max_drawdown']:.4f}."
    ),
})

diagnostic_rows.append({
    "question": "Which AURORA purged policy is best?",
    "finding": str(best_aurora_oos["policy_name"]),
    "evidence": (
        f"total_return={best_aurora_oos['total_return']:.4f}, "
        f"Sharpe={best_aurora_oos['sharpe_ratio']:.4f}, "
        f"max_drawdown={best_aurora_oos['max_drawdown']:.4f}."
    ),
})

diagnostic_df = pd.DataFrame(diagnostic_rows)
diagnostic_df.to_csv(TABLE_RUN_DIR / "notebook08_diagnostic_summary.csv", index=False)
diagnostic_df.to_csv(TABLE_DIR / f"table_65_notebook08_diagnostic_summary_{RUN_ID}.csv", index=False)

print(diagnostic_df.to_string(index=False))

# ============================================================
# 15. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Saving validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "08_AURORA_stronger_financial_baselines_and_purged_allocation.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "notebook07_run_id": NOTEBOOK07_RUN_ID,
    "notebook07_root": str(NOTEBOOK07_ROOT),
    "notebook08_input_index": str(NOTEBOOK08_INPUT_INDEX),
    "etf_return_panel": str(etf_return_path),
    "etf_close_panel": str(etf_close_path) if etf_close_path is not None else "return_based_price_proxy",
    "etf_universe": ETF_UNIVERSE,
    "oos_period": {
        "n_dates": int(len(oos_dates)),
        "start": str(oos_dates.min().date()),
        "end": str(oos_dates.max().date()),
    },
    "strict_test_period": {
        "n_dates": int(len(test_dates)),
        "start": str(test_dates.min().date()),
        "end": str(test_dates.max().date()),
    },
    "transaction_cost_rate": TRANSACTION_COST_RATE,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "aurora_configs": AURORA_CONFIGS,
    "n_policies": int(len(policy_weight_dict)),
    "n_oos_metric_rows": int(len(performance_df)),
    "n_test_metric_rows": int(len(test_performance_df)),
    "best_aurora_oos": best_aurora_oos.to_dict(),
    "best_benchmark_oos": best_benchmark_oos.to_dict(),
    "best_overall_oos": best_all_oos.to_dict(),
    "diagnostics": diagnostic_df.to_dict(orient="records"),
    "educational_note": (
        "This notebook performs research backtests only and does not provide personalized financial advice."
    ),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "weights": str(WEIGHT_DIR),
        "returns": str(RETURN_DIR),
        "plots": str(PLOT_DIR),
        "tables": str(TABLE_RUN_DIR),
        "diagnostics": str(DIAGNOSTIC_DIR),
    },
}

validation_report_path = REPORT_RUN_DIR / "AURORA_08_stronger_baselines_validation_report.json"
validation_report_global_path = REPORT_DIR / f"AURORA_08_stronger_baselines_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "AURORA_08_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_08_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 16. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 08 COMPLETE")
print("=" * 80)
print("Run ID                         :", RUN_ID)
print("Run root                       :", RUN_ROOT)
print("OOS performance table          :", TABLE_DIR / f"table_59_stronger_baseline_allocation_performance_oos_{RUN_ID}.csv")
print("Test-only performance table    :", TABLE_DIR / f"table_60_stronger_baseline_allocation_performance_test_only_{RUN_ID}.csv")
print("OOS ranking table              :", TABLE_DIR / f"table_61_allocation_rankings_oos_{RUN_ID}.csv")
print("Test-only ranking table        :", TABLE_DIR / f"table_62_allocation_rankings_test_only_{RUN_ID}.csv")
print("AURORA vs benchmark table      :", TABLE_DIR / f"table_63_aurora_vs_best_benchmark_{RUN_ID}.csv")
print("Policy descriptions table      :", TABLE_DIR / f"table_64_policy_descriptions_stronger_baselines_{RUN_ID}.csv")
print("Diagnostic summary table       :", TABLE_DIR / f"table_65_notebook08_diagnostic_summary_{RUN_ID}.csv")
print("Returns directory              :", RETURN_DIR)
print("Weights directory              :", WEIGHT_DIR)
print("Plots directory                :", PLOT_DIR)
print("Paper figures directory        :", PAPER_FIGURE_DIR)
print("Validation report              :", validation_report_path)
print("Manifest                       :", manifest_path)
print("=" * 80)

print("\nRecommended next notebook:")
print("09_AURORA_validation_optimized_regime_templates.ipynb")

Mounted at /content/drive
AURORA-TWETF Notebook 08: Stronger Baselines and Purged Allocation
Timestamp UTC       : 2026-06-24T03:42:04Z
Run ID              : 20260624_034204
Notebook 07 root    : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817
Notebook 08 registry: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv
Run root            : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/stronger_financial_baselines_purged_allocation/run_20260624_034204

Step 1: Loading ETF panels and Notebook 07 probabilities
ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
ETF return shape: (1426, 4)
ETF date range  : 2021-01-01 to 2026-06-23
ETF close panel : /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_close_panel.parquet

Notebook 08 input index:
         run_id            